# Run Coding Assistant — Dev Spaces + Open WebUI

Same coding assistant experience as `2_run_local_coding_assistant.ipynb`, but running entirely inside the cluster.

| Component | What it does |
|-----------|-------------|
| **Dev Spaces** | Browser-based IDE workspace on OpenShift |
| **OpenCode** | AI coding agent (TUI + Web UI), pre-installed in workspace |
| **Open WebUI** | Chat interface for quick model interaction |

**Prerequisites:** Dev Spaces operator installed, model deployed (Phase 0 complete)

## 1. Setup

In [4]:
%%bash
source ../.env 2>/dev/null
CLUSTER_DOMAIN=${CLUSTER_DOMAIN:-$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')}

echo "=== Dev Spaces ==="
if oc get csv -A --no-headers 2>/dev/null | grep -q devspaces; then
    echo "  Operator: installed"
else
    echo "  Operator: NOT installed — installing..."
    oc apply -f - <<'YAML'
apiVersion: operators.coreos.com/v1alpha1
kind: Subscription
metadata:
  name: devspaces
  namespace: openshift-operators
spec:
  channel: stable
  installPlanApproval: Automatic
  name: devspaces
  source: redhat-operators
  sourceNamespace: openshift-marketplace
YAML

    # If InstallPlan is pending manual approval, auto-approve it
    sleep 5
    IP=$(oc get subscription devspaces -n openshift-operators -o jsonpath='{.status.installPlanRef.name}' 2>/dev/null)
    if [ -n "$IP" ]; then
        APPROVED=$(oc get installplan $IP -n openshift-operators -o jsonpath='{.spec.approved}' 2>/dev/null)
        if [ "$APPROVED" != "true" ]; then
            echo "  Approving InstallPlan: $IP"
            oc patch installplan $IP -n openshift-operators --type merge -p '{"spec":{"approved":true}}'
        fi
    fi

    echo "  Waiting for operator CSV to reach Succeeded..."
    for i in $(seq 1 30); do
        STATUS=$(oc get csv -A --no-headers 2>/dev/null | grep devspaces | awk '{print $NF}')
        if [ "$STATUS" = "Succeeded" ]; then
            echo "  Operator installed successfully."
            break
        fi
        echo "  [$i/30] Status: ${STATUS:-Pending} — waiting 10s..."
        sleep 10
    done

    if [ "$STATUS" != "Succeeded" ]; then
        echo "  Timeout — check: oc get csv -A | grep devspaces"
    fi
fi

echo "  Dashboard: https://devspaces.${CLUSTER_DOMAIN}"

echo ""
echo "=== Model ==="
echo "  Endpoint: ${MODEL_ENDPOINT}"
echo "  Name:     ${MODEL_NAME}"

=== Dev Spaces ===


  Operator: NOT installed — installing...
subscription.operators.coreos.com/devspaces unchanged
  Waiting for operator CSV to reach Succeeded...
  [1/30] Status: Pending — waiting 10s...
  [2/30] Status: Pending — waiting 10s...
  [3/30] Status: Pending — waiting 10s...
  [4/30] Status: Pending — waiting 10s...
  [5/30] Status: Pending — waiting 10s...
  [6/30] Status: Pending — waiting 10s...
  [7/30] Status: Pending — waiting 10s...
  [8/30] Status: Pending — waiting 10s...
  [9/30] Status: Pending — waiting 10s...
  [10/30] Status: Pending — waiting 10s...
  [11/30] Status: Pending — waiting 10s...
  [12/30] Status: Pending — waiting 10s...
  [13/30] Status: Pending — waiting 10s...
  [14/30] Status: Pending — waiting 10s...
  [15/30] Status: Pending — waiting 10s...
  [16/30] Status: Pending — waiting 10s...
  [17/30] Status: Pending — waiting 10s...
  [18/30] Status: Pending — waiting 10s...
  [19/30] Status: Pending — waiting 10s...
  [20/30] Status: Pending — waiting 10s...
  [2

## 2. Create Dev Spaces Workspace

Open the **Dev Spaces Dashboard** and paste this repo URL:

```
https://github.com/hyogrin/rhoai-coding-assistant-lab
```

Dev Spaces reads [`devfile.yaml`](devfile.yaml) and creates a workspace with:
- OpenCode TUI + Web UI (port 4096)
- VS Code with OpenCode extensions
- This repo cloned into `/projects`

After the workspace starts, set the model endpoint in the workspace terminal:

In [ ]:
%%bash
source ../.env 2>/dev/null
CLUSTER_DOMAIN=${CLUSTER_DOMAIN:-$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')}
MODEL_NAME=${MODEL_NAME:-qwen36-35b}
MODEL_NS=${MODEL_NAMESPACE:-demo}
MAAS_KEY=${MAAS_API_KEY:-sk-replace-me}

cat <<EOF
=== Paste into Dev Spaces terminal ===

export OPENAI_BASE_URL="https://maas-api.${CLUSTER_DOMAIN}/${MODEL_NS}/${MODEL_NAME}/v1"
export OPENAI_API_KEY="${MAAS_KEY}"
export NODE_TLS_REJECT_UNAUTHORIZED=0

# Then start OpenCode:
opencode
EOF

## 3. Configure MCP Servers (Optional)

Inside Dev Spaces, MCP servers are accessible via cluster-internal URLs — no TLS issues, lower latency.

Add to `opencode.json` in the workspace:

In [ ]:
%%bash
cat <<'EOF'
=== Add to opencode.json ===

{
  "mcpServers": {
    "context7":        { "url": "http://mcp-context7.mcp-servers.svc.cluster.local:3001/mcp" },
    "searxng":         { "url": "http://mcp-searxng.mcp-servers.svc.cluster.local:8000/mcp" },
    "code-sandbox":    { "url": "http://mcp-code-sandbox.mcp-servers.svc.cluster.local:3005/mcp" },
    "codebase-search": { "url": "http://mcp-codebase-search.mcp-servers.svc.cluster.local:8000/mcp" },
    "repo-docs":       { "url": "http://mcp-repo-docs.mcp-servers.svc.cluster.local:8000/mcp" }
  }
}
EOF

## 4. Deploy Open WebUI

Web chat interface for quick model interaction — useful for demos and testing.

Source: [RHOAI-Toolkit/open-webui-demo](https://github.com/hyogrin/RHOAI-Toolkit/tree/main/demo/open-webui-demo)

In [ ]:
%%bash
source ../.env 2>/dev/null
CLUSTER_DOMAIN=${CLUSTER_DOMAIN:-$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')}

export NAMESPACE=${MODEL_NAMESPACE:-demo}
export MODEL_URL=${MODEL_ENDPOINT:-https://maas-api.${CLUSTER_DOMAIN}/${NAMESPACE}/${MODEL_NAME}/v1}
export API_KEY=${MAAS_API_KEY:-}

echo "Deploying Open WebUI..."
echo "  Namespace:  ${NAMESPACE}"
echo "  Model URL:  ${MODEL_URL}"

envsubst < manifests/open-webui.yaml | oc apply -f -

echo ""
echo "Waiting for rollout..."
oc rollout status deployment/open-webui -n ${NAMESPACE} --timeout=120s 2>/dev/null || true

WEBUI_URL=$(oc get route open-webui -n ${NAMESPACE} -o jsonpath='https://{.spec.host}' 2>/dev/null)
echo ""
echo "Open WebUI: ${WEBUI_URL}"

## 5. Test

Try the same task from `2_run_local_coding_assistant.ipynb`:

```
Add a GET /api/specials endpoint that returns today's daily special menu items
with discounted prices. Follow the conventions in AGENTS.md.
```

| | Local (notebook 2) | Dev Spaces (this notebook) | Open WebUI |
|--|-------------------|---------------------------|------------|
| **Runs on** | Laptop | Cluster (browser) | Cluster (browser) |
| **Interface** | VS Code / Cursor / Claude Code / OpenCode | OpenCode TUI or Web UI | Chat UI |
| **MCP access** | Via Routes (HTTPS) | Cluster-internal (HTTP) | N/A |
| **TLS setup** | Certificate workarounds needed | None | None |
| **Best for** | Daily development | Team onboarding, demos | Quick testing, non-developers |

## Cleanup (Optional)

In [ ]:
%%bash
source ../.env 2>/dev/null
NAMESPACE=${MODEL_NAMESPACE:-demo}

echo "Removing Open WebUI from ${NAMESPACE}..."
oc delete deployment,svc,route,configmap,pvc -l app=open-webui -n ${NAMESPACE} --ignore-not-found
oc delete configmap openwebui-config -n ${NAMESPACE} --ignore-not-found
oc delete pvc open-webui-data -n ${NAMESPACE} --ignore-not-found
echo "Done."